# Anahtar-değer verilerini diskte saklama

`shared_preferences` paketini kullanarak anahtar-değer (key-value)
verilerinin nasıl saklanacağını öğrenin.

Kaydedilecek nispeten küçük bir anahtar-değer koleksiyonunuz varsa,
`shared_preferences` eklentisini kullanabilirsiniz.

Normalde, verileri her platformda saklamak için yerel platform
entegrasyonları yazmanız gerekir. Neyse ki, `shared_preferences`
eklentisi, Flutter’ın desteklediği her platformda anahtar-değer
verilerini diskte kalıcı hale getirmek için kullanılabilir.

Bu tarif aşağıdaki adımları kullanır:

1.  Bağımlılığı ekleyin.
2.  Verileri kaydedin.
3.  Verileri okuyun.
4.  Verileri silin.

> **Not:** Daha fazla bilgi edinmek için `shared_preferences` paketi
> hakkındaki bu kısa **Haftanın Paketi** videosunu izleyin.

## 1. Bağımlılığı ekleyin

Başlamadan önce, `shared_preferences` paketini bağımlılık olarak
ekleyin.

`shared_preferences` paketini bağımlılık olarak eklemek için
`flutter pub add` komutunu çalıştırın:

``` bash
flutter pub add shared_preferences
```

## 2. Verileri kaydedin

Verileri kalıcı hale getirmek için `SharedPreferences` sınıfı tarafından
sağlanan ayarlayıcı (setter) yöntemleri kullanın. `setInt`, `setBool` ve
`setString` gibi çeşitli ilkel (primitive) türler için ayarlayıcı
yöntemler mevcuttur.

Ayarlayıcı yöntemler iki şey yapar: İlk olarak, bellekteki anahtar-değer
çiftini eşzamanlı (synchronously) olarak günceller. Ardından, verileri
diske kalıcı olarak yazar.

``` dart
// Bu uygulama için paylaşılan tercihleri yükleyin ve edinin.
final prefs = await SharedPreferences.getInstance();

// Sayaç değerini 'counter' anahtarı altında kalıcı depolamaya kaydedin.
await prefs.setInt('counter', counter);
```

## 3. Verileri okuyun

Verileri okumak için, `SharedPreferences` sınıfı tarafından sağlanan
uygun alıcı (getter) yöntemini kullanın. Her ayarlayıcı için karşılık
gelen bir alıcı vardır. Örneğin, `getInt`, `getBool` ve `getString`
yöntemlerini kullanabilirsiniz.

``` dart
final prefs = await SharedPreferences.getInstance();

// Kalıcı depolamadan sayaç değerini okumayı deneyin.
// Mevcut değilse, null döndürülür, bu nedenle varsayılan olarak 0 alınır.
final counter = prefs.getInt('counter') ?? 0;
```

Alıcı yöntemlerin, kalıcı değerin alıcı yönteminin beklediğinden farklı
bir türe sahip olması durumunda bir istisna (exception) fırlattığını
unutmayın.

## 4. Verileri silin

Verileri silmek için `remove()` yöntemini kullanın.

``` dart
final prefs = await SharedPreferences.getInstance();

// Sayaç anahtar-değer çiftini kalıcı depolamadan kaldırın.
await prefs.remove('counter');
```

## Desteklenen türler

`shared_preferences` tarafından sağlanan anahtar-değer depolaması
kullanımı kolay ve kullanışlı olsa da sınırlamaları vardır:

- Yalnızca ilkel türler kullanılabilir: `int`, `double`, `bool`,
  `String` ve `List<String>`.
- Büyük miktarda veri depolamak için tasarlanmamıştır.
- Uygulama yeniden başlatıldığında verilerin kalıcı olacağının garantisi
  yoktur (işletim sistemi tarafından temizlenebilir, ancak bu nadirdir).

## Test desteği

`shared_preferences` kullanarak verileri kalıcı hale getiren kodu test
etmek iyi bir fikirdir. Bunu etkinleştirmek için paket, tercih deposunun
bellek içi sahte (mock) bir uygulamasını sağlar.

Testlerinizi sahte uygulamayı kullanacak şekilde ayarlamak için, test
dosyalarınızdaki bir `setUpAll()` yönteminde `setMockInitialValues`
statik yöntemini çağırın. Başlangıç değerleri olarak kullanılacak bir
anahtar-değer çifti haritası iletin.

``` dart
SharedPreferences.setMockInitialValues(<String, Object>{'counter': 2});
```

## Tam örnek

``` dart
import 'package:flutter/material.dart';
import 'package:shared_preferences/shared_preferences.dart';

void main() => runApp(const MyApp());

class MyApp extends StatelessWidget {
  const MyApp({super.key});

  @override
  Widget build(BuildContext context) {
    return const MaterialApp(
      title: 'Shared preferences demosu',
      home: MyHomePage(title: 'Shared preferences demosu'),
    );
  }
}

class MyHomePage extends StatefulWidget {
  const MyHomePage({super.key, required this.title});

  final String title;

  @override
  State<MyHomePage> createState() => _MyHomePageState();
}

class _MyHomePageState extends State<MyHomePage> {
  int _counter = 0;

  @override
  void initState() {
    super.initState();
    _loadCounter();
  }

  /// Başlangıçta kalıcı depolamadan ilk sayaç değerini yükleyin,
  /// veya mevcut değilse 0'a geri dönün.
  Future<void> _loadCounter() async {
    final prefs = await SharedPreferences.getInstance();
    setState(() {
      _counter = prefs.getInt('counter') ?? 0;
    });
  }

  /// Bir tıklamadan sonra sayaç durumunu artırın ve
  /// asenkron olarak kalıcı depolamaya kaydedin.
  Future<void> _incrementCounter() async {
    final prefs = await SharedPreferences.getInstance();
    setState(() {
      _counter = (prefs.getInt('counter') ?? 0) + 1;
      prefs.setInt('counter', _counter);
    });
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: Text(widget.title)),
      body: Center(
        child: Column(
          mainAxisAlignment: MainAxisAlignment.center,
          children: [
            const Text('Düğmeye bu kadar çok kez bastınız: '),
            Text(
              '$_counter',
              style: Theme.of(context).textTheme.headlineMedium,
            ),
          ],
        ),
      ),
      floatingActionButton: FloatingActionButton(
        onPressed: _incrementCounter,
        tooltip: 'Artır',
        child: const Icon(Icons.add),
      ),
    );
  }
}
```

# Dosyaları okuma ve yazma

Bazı durumlarda, diskteki dosyaları okumanız ve yazmanız gerekir.
Örneğin, uygulama başlatmaları arasında verileri kalıcı hale getirmeniz
veya internetten veri indirip daha sonra çevrimdışı kullanım için
kaydetmeniz gerekebilir.

Mobil veya masaüstü uygulamalarında dosyaları diske kaydetmek için,
`path_provider` eklentisini `dart:io` kütüphanesi ile birleştirin.

Bu tarif (yöntem) aşağıdaki adımları kullanır: 1. Doğru yerel yolu
bulun. 2. Dosya konumuna bir referans oluşturun. 3. Veriyi dosyaya
yazın. 4. Veriyi dosyadan okuyun.

Daha fazlasını öğrenmek için `path_provider` paketi hakkındaki bu
“Haftanın Paketi” videosunu izleyin:

*(Video içeriği)*

> **Not** Bu tarif şu anda web uygulamalarıyla çalışmaz. Bu konudaki
> tartışmayı takip etmek için flutter/flutter sorun #45296’ya göz atın.

## 1. Doğru yerel yolu bulun

Bu örnek bir sayaç görüntüler. Sayaç değiştiğinde, uygulama
yüklendiğinde tekrar okuyabilmek için veriyi diske yazın. Peki bu veriyi
nerede saklamalısınız?

`path_provider` paketi, cihazın dosya sisteminde sık kullanılan
konumlara erişmek için platformdan bağımsız bir yol sağlar. Eklenti şu
anda iki dosya sistemi konumuna erişimi destekler:

- **Geçici dizin (Temporary directory):** Sistemin herhangi bir zamanda
  temizleyebileceği geçici bir dizin (önbellek). iOS’te bu
  `NSCachesDirectory`’ye karşılık gelir. Android’de bu, `getCacheDir()`
  metodunun döndürdüğü değerdir.
- **Belgeler dizini (Documents directory):** Uygulamanın yalnızca
  kendisinin erişebileceği dosyaları saklaması için bir dizin. Sistem bu
  dizini yalnızca uygulama silindiğinde temizler. iOS’te bu
  `NSDocumentDirectory`’ye karşılık gelir. Android’de bu `AppData`
  dizinidir.

Bu örnek, bilgileri belgeler dizininde saklar. Belgeler dizininin yolunu
şu şekilde bulabilirsiniz:

``` dart
import 'package:path_provider/path_provider.dart';
  // ···
  Future<String> get _localPath async {
    final directory = await getApplicationDocumentsDirectory();

    return directory.path;
  }
```

## 2. Dosya konumuna bir referans oluşturun

Dosyayı nerede saklayacağınızı bildiğinizde, dosyanın tam konumuna bir
referans oluşturun. Bunu başarmak için `dart:io` kütüphanesinden `File`
sınıfını kullanabilirsiniz.

``` dart
Future<File> get _localFile async {
  final path = await _localPath;
  return File('$path/counter.txt');
}
```

## 3. Veriyi dosyaya yazın

Artık çalışacak bir `File` (Dosya) nesneniz olduğuna göre, verileri
okumak ve yazmak için onu kullanın. Önce, dosyaya biraz veri yazın.
Sayaç bir tamsayıdır ancak `'$counter'` sözdizimi kullanılarak dosyaya
bir dize (string) olarak yazılır.

``` dart
Future<File> writeCounter(int counter) async {
  final file = await _localFile;

  // Dosyayı yaz
  return file.writeAsString('$counter');
}
```

## 4. Veriyi dosyadan okuyun

Artık diskte bazı verileriniz olduğuna göre, bunları okuyabilirsiniz.
Bir kez daha `File` sınıfını kullanın.

``` dart
Future<int> readCounter() async {
  try {
    final file = await _localFile;

    // Dosyayı oku
    final contents = await file.readAsString();

    return int.parse(contents);
  } catch (e) {
    // Bir hatayla karşılaşılırsa 0 döndür
    return 0;
  }
}
```

## Tam Örnek

``` dart
import 'dart:async';
import 'dart:io';

import 'package:flutter/material.dart';
import 'package:path_provider/path_provider.dart';

void main() {
  runApp(
    MaterialApp(
      title: 'Dosya Okuma ve Yazma',
      home: FlutterDemo(storage: CounterStorage()),
    ),
  );
}

class CounterStorage {
  Future<String> get _localPath async {
    final directory = await getApplicationDocumentsDirectory();

    return directory.path;
  }

  Future<File> get _localFile async {
    final path = await _localPath;
    return File('$path/counter.txt');
  }

  Future<int> readCounter() async {
    try {
      final file = await _localFile;

      // Dosyayı oku
      final contents = await file.readAsString();

      return int.parse(contents);
    } catch (e) {
      // Bir hatayla karşılaşılırsa 0 döndür
      return 0;
    }
  }

  Future<File> writeCounter(int counter) async {
    final file = await _localFile;

    // Dosyayı yaz
    return file.writeAsString('$counter');
  }

}

class FlutterDemo extends StatefulWidget {
  const FlutterDemo({super.key, required this.storage});

  final CounterStorage storage;

  @override
  State<FlutterDemo> createState() => _FlutterDemoState();
}

class _FlutterDemoState extends State<FlutterDemo> {
  int _counter = 0;

  @override
  void initState() {
    super.initState();
    widget.storage.readCounter().then((value) {
      setState(() {
        _counter = value;
      });
    });
  }

  Future<File> _incrementCounter() {
    setState(() {
      _counter++;
    });

    // Değişkeni bir dize (string) olarak dosyaya yaz.
    return widget.storage.writeCounter(_counter);
  }

  @override
  Widget build(BuildContext context) {
    return Scaffold(
      appBar: AppBar(title: const Text('Dosya Okuma ve Yazma')),
      body: Center(
        child: Text('Düğmeye $_counter kez basıldı.'),
      ),
      floatingActionButton: FloatingActionButton(
        onPressed: _incrementCounter,
        tooltip: 'Artır',
        child: const Icon(Icons.add),
      ),
    );
  }
}
```

# Verileri SQLite ile kalıcı hale getirme

Verileri depolamak ve geri çağırmak için SQLite nasıl kullanılır.

> **Not:** Bu kılavuz `sqflite` paketini kullanır. Bu paket yalnızca
> macOS, iOS veya Android üzerinde çalışan uygulamaları destekler.

Yerel cihazda büyük miktarda veriyi kalıcı hale getirmesi ve sorgulaması
gereken bir uygulama yazıyorsanız, yerel bir dosya veya anahtar-değer
deposu (key-value store) yerine bir veritabanı kullanmayı düşünün. Genel
olarak veritabanları, diğer yerel kalıcılık çözümlerine kıyasla daha
hızlı ekleme, güncelleme ve sorgulama sağlar.

Flutter uygulamaları, pub.dev üzerinde bulunan `sqflite` eklentisi
aracılığıyla SQLite veritabanlarını kullanabilir. Bu tarif, çeşitli
Köpekler (Dogs) hakkındaki verileri eklemek, okumak, güncellemek ve
kaldırmak için `sqflite` kullanımının temellerini gösterir.

Eğer SQLite ve SQL ifadelerinde yeniyseniz, bu tarifi tamamlamadan önce
temelleri öğrenmek için **SQLite Eğitimi**’ni inceleyin.

Bu tarif aşağıdaki adımları kullanır:

1.  Bağımlılıkları ekleyin.
2.  `Dog` veri modelini tanımlayın.
3.  Veritabanını açın.
4.  `dogs` tablosunu oluşturun.
5.  Veritabanına bir `Dog` ekleyin.
6.  Köpeklerin listesini alın (Retrieve).
7.  Veritabanındaki bir `Dog`’u güncelleyin.
8.  Veritabanından bir `Dog` silin.

## 1. Bağımlılıkları ekleyin

SQLite veritabanlarıyla çalışmak için `sqflite` ve `path` paketlerini
içe aktarın.

- `sqflite` paketi, bir SQLite veritabanıyla etkileşim kurmak için
  sınıflar ve işlevler sağlar.
- `path` paketi, veritabanını diskte saklamak için konumu tanımlayan
  işlevler sağlar.

Paketleri bağımlılık olarak eklemek için `flutter pub add` komutunu
çalıştırın:

``` bash
flutter pub add sqflite path
```

Çalışacağınız dosyada paketleri içe aktardığınızdan emin olun.

``` dart
import 'dart:async';

import 'package:flutter/widgets.dart';
import 'package:path/path.dart';
import 'package:sqflite/sqflite.dart';
```

## 2. Dog veri modelini tanımlayın

Köpekler hakkındaki bilgileri saklayacak tabloyu oluşturmadan önce,
saklanması gereken verileri tanımlamak için biraz zaman ayırın. Bu örnek
için, üç parça veri içeren bir `Dog` sınıfı tanımlayın: Benzersiz bir
`id`, `name` (ad) ve her köpeğin `age` (yaşı).

``` dart
class Dog {
  final int id;
  final String name;
  final int age;

  const Dog({required this.id, required this.name, required this.age});
}
```

## 3. Veritabanını açın

Veritabanına veri okumadan ve yazmadan önce, veritabanına bir bağlantı
açın. Bu iki adım içerir:

1.  `sqflite` paketinden `getDatabasesPath()` ve `path` paketinden
    `join` işlevini kullanarak veritabanı dosyasının yolunu tanımlayın.
2.  `sqflite` paketinden `openDatabase()` işlevi ile veritabanını açın.

> **Not:** `await` anahtar kelimesini kullanabilmek için kodun bir
> `async` işlevi içine yerleştirilmesi gerekir. Aşağıdaki tüm tablo
> işlevlerini `void main() async {}` içine yerleştirmelisiniz.

``` dart
// flutter upgrade kaynaklı hatalardan kaçının.
// 'package:flutter/widgets.dart' paketini içe aktarmak gereklidir.
WidgetsFlutterBinding.ensureInitialized();
// Veritabanını açın ve referansı saklayın.
final database = openDatabase(
  // Veritabanının yolunu ayarlayın. Not: `path` paketinden `join` işlevini
  // kullanmak, yolun her platform için doğru şekilde oluşturulmasını sağlamak
  // adına en iyi uygulamadır.
  join(await getDatabasesPath(), 'doggie_database.db'),
);
```

## 4. `dogs` tablosunu oluşturun

Ardından, çeşitli Köpekler hakkında bilgi saklamak için bir tablo
oluşturun. Bu örnek için, saklanabilecek verileri tanımlayan `dogs`
adında bir tablo oluşturun. Her `Dog`; bir `id`, `name` ve `age` içerir.
Bu nedenle, bunlar `dogs` tablosunda üç sütun olarak temsil edilir.

- `id` bir Dart `int`’idir ve `INTEGER` SQLite Veri Tipi olarak
  saklanır. Sorgulama ve güncelleme sürelerini iyileştirmek için `id`’yi
  tablonun birincil anahtarı (primary key) olarak kullanmak iyi bir
  uygulamadır.
- `name` bir Dart `String`’idir ve `TEXT` SQLite Veri Tipi olarak
  saklanır.
- `age` de bir Dart `int`’idir ve `INTEGER` Veri Tipi olarak saklanır.

Bir SQLite veritabanında saklanabilecek mevcut Veri Tipleri hakkında
daha fazla bilgi için resmi **SQLite Veri Tipleri belgelerine** bakın.

``` dart
final database = openDatabase(
  // Veritabanının yolunu ayarlayın.
  join(await getDatabasesPath(), 'doggie_database.db'),
  // Veritabanı ilk oluşturulduğunda, köpekleri saklamak için bir tablo oluşturun.
  onCreate: (db, version) {
    // Veritabanında CREATE TABLE ifadesini çalıştırın.
    return db.execute(
      'CREATE TABLE dogs(id INTEGER PRIMARY KEY, name TEXT, age INTEGER)',
    );
  },
  // Sürümü ayarlayın. Bu, onCreate işlevini yürütür ve veritabanı
  // yükseltmeleri ve düşürmeleri (upgrades/downgrades) için bir yol sağlar.
  version: 1,
);
```

## 5. Veritabanına bir Dog ekleyin

Artık çeşitli köpekler hakkında bilgi saklamaya uygun bir tablosu olan
bir veritabanınız olduğuna göre, veri okuma ve yazma zamanı geldi.

İlk olarak, `dogs` tablosuna bir `Dog` ekleyin. Bu iki adım içerir:

1.  `Dog`’u bir `Map`’e dönüştürün.
2.  `Map`’i `dogs` tablosuna saklamak için `insert()` yöntemini
    kullanın.

``` dart
class Dog {
  final int id;
  final String name;
  final int age;

  Dog({required this.id, required this.name, required this.age});

  // Bir Dog'u Map'e dönüştürün. Anahtarlar (keys), veritabanındaki
  // sütun adlarına karşılık gelmelidir.
  Map<String, Object?> toMap() {
    return {'id': id, 'name': name, 'age': age};
  }

  // print ifadesini kullanırken her köpek hakkındaki bilgileri
  // görmeyi kolaylaştırmak için toString'i uygulayın.
  @override
  String toString() {
    return 'Dog{id: $id, name: $name, age: $age}';
  }
}

// Veritabanına köpekleri ekleyen bir fonksiyon tanımlayın
Future<void> insertDog(Dog dog) async {
  // Veritabanı referansını alın.
  final db = await database;

  // Dog'u doğru tabloya ekleyin. Aynı köpeğin iki kez eklenmesi durumunda
  // kullanılacak `conflictAlgorithm`'i de belirtebilirsiniz.
  //
  // Bu durumda, önceki verileri değiştirin.
  await db.insert(
    'dogs',
    dog.toMap(),
    conflictAlgorithm: ConflictAlgorithm.replace,
  );
}

// Bir Dog oluşturun ve dogs tablosuna ekleyin
var fido = Dog(id: 0, name: 'Fido', age: 35);

await insertDog(fido);
```

## 6. Köpeklerin listesini alın

Veritabanında bir `Dog` saklandığına göre, belirli bir köpeği veya tüm
köpeklerin listesini sorgulayın. Bu iki adım içerir:

1.  `dogs` tablosuna karşı bir `query` (sorgu) çalıştırın. Bu bir
    `List<Map>` döndürür.
2.  `List<Map>`’i bir `List<Dog>`’a dönüştürün.

``` dart
// dogs tablosundaki tüm köpekleri getiren bir yöntem.
Future<List<Dog>> dogs() async {
  // Veritabanı referansını alın.
  final db = await database;

  // Tüm köpekler için tabloyu sorgulayın.
  final List<Map<String, Object?>> dogMaps = await db.query('dogs');

  // Her köpeğin alanlarını içeren listeyi `Dog` nesneleri listesine dönüştürün.
  return [
    for (final {'id': id as int, 'name': name as String, 'age': age as int}
        in dogMaps)
      Dog(id: id, name: name, age: age),
  ];
}

// Şimdi, tüm köpekleri almak için yukarıdaki yöntemi kullanın.
print(await dogs()); // Fido'yu içeren bir liste yazdırır.
```

## 7. Veritabanındaki bir Dog’u güncelleyin

Veritabanına bilgi ekledikten sonra, bu bilgiyi daha sonra güncellemek
isteyebilirsiniz. Bunu `sqflite` kütüphanesindeki `update()` yöntemini
kullanarak yapabilirsiniz.

Bu iki adım içerir:

1.  Dog’u bir Map’e dönüştürün.
2.  Doğru Dog’u güncellediğinizden emin olmak için bir `where` cümlesi
    kullanın.

``` dart
Future<void> updateDog(Dog dog) async {
  // Veritabanı referansını alın.
  final db = await database;

  // Verilen Dog'u güncelleyin.
  await db.update(
    'dogs',
    dog.toMap(),
    // Dog'un eşleşen bir id'ye sahip olduğundan emin olun.
    where: 'id = ?',
    // SQL enjeksiyonunu önlemek için Dog'un id'sini whereArg olarak iletin.
    whereArgs: [dog.id],
  );
}

// Fido'nun yaşını güncelleyin ve veritabanına kaydedin.
fido = Dog(id: fido.id, name: fido.name, age: fido.age + 7);
await updateDog(fido);

// Güncellenen sonuçları yazdırın.
print(await dogs()); // 42 yaşındaki Fido'yu yazdırır.
```

> **Uyarı:** Bir `where` ifadesine argüman iletmek için her zaman
> `whereArgs` kullanın. Bu, SQL enjeksiyonu saldırılarına karşı
> korunmaya yardımcı olur. `where: "id = ${dog.id}"` gibi dize
> enterpolasyonları kullanmayın!

## 8. Veritabanından bir Dog silin

Köpekler hakkında bilgi ekleme ve güncellemenin yanı sıra, köpekleri
veritabanından kaldırabilirsiniz. Verileri silmek için `sqflite`
kütüphanesindeki `delete()` yöntemini kullanın.

Bu bölümde, bir id alan ve eşleşen id’ye sahip köpeği veritabanından
silen bir fonksiyon oluşturun. Bunun çalışması için, silinecek kayıtları
sınırlayan bir `where` cümlesi sağlamanız gerekir.

``` dart
Future<void> deleteDog(int id) async {
  // Veritabanı referansını alın.
  final db = await database;

  // Dog'u veritabanından kaldırın.
  await db.delete(
    'dogs',
    // Belirli bir köpeği silmek için bir `where` cümlesi kullanın.
    where: 'id = ?',
    // SQL enjeksiyonunu önlemek için Dog'un id'sini whereArg olarak iletin.
    whereArgs: [id],
  );
}
```

## Örnek

Örneği çalıştırmak için:

1.  Yeni bir Flutter projesi oluşturun.
2.  `sqflite` ve `path` paketlerini `pubspec.yaml` dosyanıza ekleyin.
3.  Aşağıdaki kodu `lib/db_test.dart` adlı yeni bir dosyaya yapıştırın.
4.  Kodu `flutter run lib/db_test.dart` komutuyla çalıştırın.

``` dart
import 'dart:async';

import 'package:flutter/widgets.dart';
import 'package:path/path.dart';
import 'package:sqflite/sqflite.dart';

void main() async {
  // flutter upgrade kaynaklı hatalardan kaçının.
  // 'package:flutter/widgets.dart' paketini içe aktarmak gereklidir.
  WidgetsFlutterBinding.ensureInitialized();
  // Veritabanını açın ve referansı saklayın.
  final database = openDatabase(
    // Veritabanının yolunu ayarlayın. Not: `path` paketinden `join` işlevini
    // kullanmak, yolun her platform için doğru şekilde oluşturulmasını sağlamak
    // adına en iyi uygulamadır.
    join(await getDatabasesPath(), 'doggie_database.db'),
    // Veritabanı ilk oluşturulduğunda, köpekleri saklamak için bir tablo oluşturun.
    onCreate: (db, version) {
      // Veritabanında CREATE TABLE ifadesini çalıştırın.
      return db.execute(
        'CREATE TABLE dogs(id INTEGER PRIMARY KEY, name TEXT, age INTEGER)',
      );
    },
    // Sürümü ayarlayın. Bu, onCreate işlevini yürütür ve veritabanı
    // yükseltmeleri ve düşürmeleri için bir yol sağlar.
    version: 1,
  );

  // Veritabanına köpekleri ekleyen bir fonksiyon tanımlayın
  Future<void> insertDog(Dog dog) async {
    // Veritabanı referansını alın.
    final db = await database;

    // Dog'u doğru tabloya ekleyin. Aynı köpeğin iki kez eklenmesi durumunda
    // kullanılacak `conflictAlgorithm`'i de belirtebilirsiniz.
    //
    // Bu durumda, önceki verileri değiştirin.
    await db.insert(
      'dogs',
      dog.toMap(),
      conflictAlgorithm: ConflictAlgorithm.replace,
    );
  }

  // dogs tablosundaki tüm köpekleri getiren bir yöntem.
  Future<List<Dog>> dogs() async {
    // Veritabanı referansını alın.
    final db = await database;

    // Tüm köpekler için tabloyu sorgulayın.
    final List<Map<String, Object?>> dogMaps = await db.query('dogs');

    // Her köpeğin alanlarını içeren listeyi `Dog` nesneleri listesine dönüştürün.
    return [
      for (final {'id': id as int, 'name': name as String, 'age': age as int}
          in dogMaps)
        Dog(id: id, name: name, age: age),
    ];
  }

  Future<void> updateDog(Dog dog) async {
    // Veritabanı referansını alın.
    final db = await database;

    // Verilen Dog'u güncelleyin.
    await db.update(
      'dogs',
      dog.toMap(),
      // Dog'un eşleşen bir id'ye sahip olduğundan emin olun.
      where: 'id = ?',
      // SQL enjeksiyonunu önlemek için Dog'un id'sini whereArg olarak iletin.
      whereArgs: [dog.id],
    );
  }

  Future<void> deleteDog(int id) async {
    // Veritabanı referansını alın.
    final db = await database;

    // Dog'u veritabanından kaldırın.
    await db.delete(
      'dogs',
      // Belirli bir köpeği silmek için bir `where` cümlesi kullanın.
      where: 'id = ?',
      // SQL enjeksiyonunu önlemek için Dog'un id'sini whereArg olarak iletin.
      whereArgs: [id],
    );
  }

  // Bir Dog oluşturun ve dogs tablosuna ekleyin
  var fido = Dog(id: 0, name: 'Fido', age: 35);

  await insertDog(fido);

  // Şimdi, tüm köpekleri almak için yukarıdaki yöntemi kullanın.
  print(await dogs()); // Fido'yu içeren bir liste yazdırır.

  // Fido'nun yaşını güncelleyin ve veritabanına kaydedin.
  fido = Dog(id: fido.id, name: fido.name, age: fido.age + 7);
  await updateDog(fido);

  // Güncellenen sonuçları yazdırın.
  print(await dogs()); // 42 yaşındaki Fido'yu yazdırır.

  // Fido'yu veritabanından silin.
  await deleteDog(fido.id);

  // Köpeklerin listesini yazdırın (boş).
  print(await dogs());
}

class Dog {
  final int id;
  final String name;
  final int age;

  Dog({required this.id, required this.name, required this.age});

  // Bir Dog'u Map'e dönüştürün. Anahtarlar, veritabanındaki
  // sütun adlarına karşılık gelmelidir.
  Map<String, Object?> toMap() {
    return {'id': id, 'name': name, 'age': age};
  }

  // print ifadesini kullanırken her köpek hakkındaki bilgileri
  // görmeyi kolaylaştırmak için toString'i uygulayın.
  @override
  String toString() {
    return 'Dog{id: $id, name: $name, age: $age}';
  }
}
```

## 📄 Lisans Bilgisi

Bu doküman, **Flutter resmi dokümantasyonundan** türetilmiş Türkçe ders
notudur.

**Orijinal kaynak:**  
https://docs.flutter.dev/cookbook/persistence/key-value#3-read-data

**Türkçe çeviri ve düzenleme:**  
[Doç. Dr. Hakan Temiz](mailto:htemiz@artvin.edu.tr)

------------------------------------------------------------------------

### Lisans Kapsamı

Bu dokümandaki içerikler aşağıdaki açık lisanslar kapsamında
sunulmaktadır:

**Metin içerikleri (anlatım ve açıklamalar):**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** Creative Commons Attribution 4.0 International (CC BY 4.0)  
https://creativecommons.org/licenses/by/4.0/

Bu lisans kapsamında: - İçerik kopyalanabilir, dağıtılabilir ve
uyarlanabilir  
- Ticari kullanım serbesttir  
- Kaynak belirtilmesi zorunludur

**Kod örnekleri:**  
Flutter resmi dokümantasyonundan alınmış veya uyarlanmıştır.  
**Lisans:** BSD 3-Clause License  
https://opensource.org/licenses/BSD-3-Clause

Bu lisans kapsamında: - Kodlar kopyalanabilir, değiştirilebilir ve
dağıtılabilir  
- Ticari kullanım serbesttir  
- Lisans bildiriminin korunması gerekir